In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from io import BytesIO

In [ ]:
class czastka:
  def __init__(self, pos, vel):
    self.r = pos
    self.v = vel

In [ ]:
def draw_with_ghosts(ax, pos_all, box_size, r=0.3, s=1160, color='royalblue'):
    ax.scatter(pos_all[:, 0], pos_all[:, 1], s=s, color=color)

    ghosts = []
    left  = pos_all[pos_all[:, 0] < r] + [box_size, 0]
    right = pos_all[pos_all[:, 0] > box_size - r] - [box_size, 0]
    bottom = pos_all[pos_all[:, 1] < r] + [0, box_size]
    top    = pos_all[pos_all[:, 1] > box_size - r] - [0, box_size]

    ghosts += [left, right, bottom, top]
    ghosts = [g for g in ghosts if len(g)]
    if ghosts:
        g = np.vstack(ghosts)
        ax.scatter(g[:, 0], g[:, 1], s=s, color=color)


In [ ]:
n = 4
box_size = 8.
eps, sigma, kB, mass = 1, 1, 1, 1
radius = sigma / 2
dt = 1e-4
T = 2.5

rC = 2.5 * sigma

In [ ]:
sigma_v = np.sqrt(kB * T / mass)
vel = np.random.normal(0, sigma_v, size=(n, n, 2))
vel -= np.mean(vel, axis=(0, 1))
T_actual = mass * np.mean(vel**2) / (2 * kB)
vel *= np.sqrt(T / T_actual)

pos = (np.array([[[i, j] for j in range(n)] for i in range(n)]) / n + np.array([1, 1]) / (2 * n)) * box_size

particles = [[czastka(pos[i, j], vel[i, j]) for j in range(n)] for i in range(n)]
flat = np.ravel(particles)

# n -> n - 1/2
for a_idx in range(len(flat)):
  for b_idx in range(a_idx + 1, len(flat)):
    i, j = flat[a_idx], flat[b_idx]
    r = i.r - j.r
    r -= box_size * np.round(r / box_size) # PBC
    dist = np.linalg.norm(r)
    if dist < rC:
      F = 48 * eps * ((sigma / dist)**12 - 0.5 * (sigma / dist)**6) * (r / dist**2)
      i.v -= F / mass * dt / 2
      j.v += F / mass * dt / 2

In [ ]:
fig, a = plt.subplots(figsize=(6, 6))
a.set_xlim(0, box_size)
a.set_ylim(0, box_size)
a.set_aspect('equal')

steps = 2*10**4 + 1
P_rC = 4 * eps * ((sigma / rC)**12 - (sigma / rC)**6)
K, P, work = [np.zeros((steps)) for _ in range(3)]

with imageio.get_writer('../results/gifs/05_molecular_dynamics_gas.gif', mode='I', duration=0.1) as writer:
  for k in range(steps): # 10**4 -> 30s
    prev_v = np.array([p.v.copy() for p in flat])

    for a_idx in range(len(flat)):
      for b_idx in range(a_idx + 1, len(flat)):
        i, j = flat[a_idx], flat[b_idx]
        r = i.r - j.r
        r -= box_size * np.round(r / box_size) # PBC
        dist = np.linalg.norm(r)

        if dist < rC:
          F = 48 * eps * ((sigma / dist)**12 - 0.5 * (sigma / dist)**6) * (r / dist**2)
          P[k] += 4 * eps * ((sigma / dist)**12 - (sigma / dist)**6) - P_rC
          work[k] += np.dot(F, r)

          i.v += F / mass * dt
          j.v -= F / mass * dt

    for idx, p in enumerate(flat):
      v_half = 0.5 * (prev_v[idx] + p.v)
      K[k] += 0.5 * mass * np.linalg.norm(v_half)**2

      p.r += p.v * dt
      p.r %= box_size

    if k % 500 == 0:
      a.clear()
      pos_all = np.array([p.r.copy() for p in flat])
      draw_with_ghosts(a, pos_all, box_size)
      a.set(xlim=(0, box_size), ylim=(0, box_size), aspect='equal',
            title=f'Symulacja gazu Lennarda-Jonesa, krok {k:05d}')
      
      buf = BytesIO()
      plt.savefig(buf, format='png')
      buf.seek(0)
      writer.append_data(imageio.imread(buf))

In [ ]:
fig, ax = plt.subplots()

ax.errorbar(np.linspace(0, 1, steps), K, label='K')
ax.errorbar(np.linspace(0, 1, steps), P, label='P')
ax.errorbar(np.linspace(0, 1, steps), K + P, label='E')

plt.savefig('../results/plots/05_gas_energy.png')
ax.legend()
plt.show()
plt.close(fig)

In [ ]:
fig, ax = plt.subplots()

pressure = (len(flat) * kB * T + work / 2) / (box_size**2)
ax.errorbar(np.linspace(0, 1, steps), K / (kB * len(flat)), label='T')
ax.errorbar(np.linspace(0, 1, steps), pressure, label='P')

plt.savefig('../results/plots/05_gas_temp_press.png')
ax.legend()
plt.show()
plt.close(fig)

In [ ]:
fig, a = plt.subplots(figsize=(6, 6))
a.set_xlim(0, box_size)
a.set_ylim(0, box_size)
a.set_aspect('equal')

with imageio.get_writer('../results/gifs/05_molecular_dynamics_gas.gif', mode='I', duration=0.1) as writer:
  for k in range(150000 + 1): # 10**4 -> 40s
    for a_idx in range(len(flat)):
      for b_idx in range(a_idx + 1, len(flat)):
        i, j = flat[a_idx], flat[b_idx]
        r = i.r - j.r
        r -= box_size * np.round(r / box_size) # PBC
        dist = np.linalg.norm(r)
        if dist < rC:
          F = 48 * eps * ((sigma / dist)**12 - 0.5 * (sigma / dist)**6) * (r / dist**2)
          i.v += F / mass * dt
          j.v -= F / mass * dt

    for p in flat:
      p.r += p.v * dt
      p.r %= box_size

    if k % 500 == 0:
      a.clear()
      pos_all = np.array([p.r for p in flat])
      a.scatter(pos_all[:, 0], pos_all[:, 1], s=1200, color='royalblue')
      a.set_xlim(0, box_size)
      a.set_ylim(0, box_size)
      a.set_aspect('equal')
      a.set_title(f'Symulacja gazu Lennarda-Jonesa, krok {k:06d}')
      
      buf = BytesIO()
      plt.savefig(buf, format='png')
      buf.seek(0)
      image = imageio.imread(buf)
      writer.append_data(image)